# Execução Passo a Passo do Experimento

Este notebook é útil quando você quer acompanhar cada etapa separadamente, repetir apenas uma parte do pipeline ou depurar uma execução específica.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time
import yaml

REPO_URL = "https://github.com/stef325/plagiarismDetectMethodsEval.git"
PROJECT_DIR = Path("/content/plagiarismDetectMethodsEval")
IN_COLAB = "google.colab" in sys.modules

def find_existing_project() -> Path | None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "main.py").exists():
            return candidate
    return None

existing_project = find_existing_project()
if existing_project is not None:
    PROJECT_DIR = existing_project
elif IN_COLAB:
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise FileNotFoundError("Repositorio nao localizado.")

os.chdir(PROJECT_DIR)
print(PROJECT_DIR)


In [ ]:
%pip install -r requirements.txt

In [ ]:
DATASET_PATH = PROJECT_DIR / "data" / "raw" / "POP909"
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        "Dataset POP909 nao encontrado em data/raw/POP909. Ajuste este caminho antes de executar."
    )

with (PROJECT_DIR / "config" / "default.yaml").open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config["dataset"]["path"] = str(DATASET_PATH)
config_path = PROJECT_DIR / "config" / "notebook_steps.yaml"
with config_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False, allow_unicode=True)

print(config_path)


In [ ]:
def run_step(step: str) -> None:
    command = [sys.executable, "src/main.py", "--config", "config/notebook_steps.yaml", step]
    print("Executando:", " ".join(command))
    start = time.time()
    subprocess.run(command, check=True)
    print(f"Tempo: {time.time() - start:.2f} segundos")

steps = [
    "inspect",
    "clean",
    "validate",
    "subset",
    "segments",
    "representations",
    "melody_transform",
    "harmony_transform",
    "rhythm_transform",
    "combined_transform",
    "validate_representations",
    "validate_transformations",
    "compute_melody_metrics",
    "compute_harmony_metrics",
    "compute_rhythm_metrics",
    "compute_global_metrics",
    "validate_metrics",
    "build_experiment_pairs",
    "run_experiment",
    "evaluate_robustness",
    "evaluate_interpretability",
    "consolidate_results",
    "generate_visualizations",
]

steps


## Executar apenas uma etapa

Escolha uma etapa da lista `steps` e execute a célula abaixo.

In [ ]:
STEP_TO_RUN = "inspect"
run_step(STEP_TO_RUN)


## Executar todas as etapas em sequência

Descomente a célula abaixo se quiser percorrer o pipeline inteiro de forma explícita.

In [ ]:
# for step in steps:
#     run_step(step)
